## Install Libraries

In [ ]:
# %pip install -q langchain==0.2.3 faiss-cpu google-generativeai langchain-core==0.2.5 langchain-community==0.2.4 langchain-google-genai jq streamlit

In [55]:
import json
from langchain_community.document_loaders import JSONLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv


load_dotenv()

True

### STEP - 1a Indexing(Document Ingestion)

In [56]:
try:
    loader = JSONLoader(
    file_path="data.json",
    jq_schema=".",
    text_content=False
    )

    data = loader.load()

    # print(data)
    data = json.loads(data[0].page_content)
    
except Exception as e:
    print(f"Error loading data: {e}")

### Text Splitting

In [57]:
chunks = []

# pricing plans
for plan in data["AutoStream"]["pricing_plans"]:
    content = f"""
    Plan: {plan['plan_name']}
    Price: ${plan['price_per_month']} per month
    Features: {plan['features']}
    """
    chunks.append(Document(page_content=content.strip()))

# policies
policies = data["AutoStream"]["company_policies"]

chunks.append(Document(
    page_content=f"Refund Policy: {policies['refund_policy']}"
))

chunks.append(Document(
    page_content=f"Support: {policies['support']}"
))

# output
for doc in chunks:
    print(doc.page_content)

Plan: Basic
    Price: $29 per month
    Features: {'videos_per_month': 10, 'resolution': '720p'}
Plan: Pro
    Price: $79 per month
    Features: {'videos_per_month': 'Unlimited', 'resolution': '4K', 'ai_captions': True}
Refund Policy: No refunds after 7 days
Support: {'availability': '24/7', 'applicable_plan': 'Pro'}


### STEP - 1a Indexing (Embedding Generation and Storing in Vector Store)

In [58]:
embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

vectorstore = FAISS.from_documents(chunks, embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [59]:
vectorstore.index_to_docstore_id

{0: 'a7f0800c-dd2e-43a5-a481-fad5b4279df8',
 1: 'a60112a9-9a88-4bcc-acd8-e5773fe4f711',
 2: 'cdbb2200-d42c-4e11-8aa6-0317b0a82cea',
 3: '3661db15-dbe5-4236-8c56-59b18ae1a3a7'}

### STEP - 2 Retrieval

In [60]:
retrieval = vectorstore.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":1, "lambda_mult": 0.5}
)

In [61]:
# Examples
retrieval.invoke("Tell me about support policy")
retrieval.invoke("What about refund ?")

[Document(id='cdbb2200-d42c-4e11-8aa6-0317b0a82cea', metadata={}, page_content='Refund Policy: No refunds after 7 days')]

### Augmentation

In [62]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [79]:
# Structure the Output for intend detection
class Intend(BaseModel):
    intend: Literal["greeting", "inquiry", "high_intend"] = Field(description="The classification of the user's intent.")
    

intent_parser = PydanticOutputParser(pydantic_object=Intend)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an intent classifier for AutoStream, a SaaS for video editing.
    Classify the user's latest message into:
    1. 'greeting': hi, hello, etc.
    2. 'inquiry': questions about pricing, features, or policies.
    3. 'high_intent': ready to sign up, buy, or try the pro plan.
    
    Answer instructions: {format_instructions}"""),
    MessagesPlaceholder(variable_name="chat_history"), 
    ("human", "{question}")
])


prompt = prompt.partial(format_instructions=intent_parser.get_format_instructions())

In [64]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are the AutoStream assistant. Use the context to answer product questions.\nContext: {context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")])

### Chat History

In [65]:
chat_history = ChatMessageHistory()

### Chains

In [84]:
# This is the intend Prompt chain
parser = StrOutputParser()
question = "what about the pricing"
intend_chain = prompt | llm | intent_parser
print(intend_chain.invoke({"chat_history": chat_history.messages, "question": question}))


intend='inquiry'


In [67]:
# Extracting the context
question = "what about the pricing"
context_text = retrieval.invoke(question)

In [86]:
# This is for rag chain
question = "Can you tell me about the pricing"
rag_chain = rag_prompt | llm | parser
print(rag_chain.invoke({"context": context_text, "history": chat_history.messages, "input": question}))

The Basic plan is priced at $29 per month.
